# Diabetes Risk Prediction from Health-Survey Data

**Seminar:** Advanced Applied Data Science  
**Institution:** Goethe University Frankfurt  
**Term:** Summer Semester 2026  
**Supervisor:** Prof. Dr. Kevin Bauer  
**Group:** Diabetes Prediction  

**Dataset:** CDC BRFSS 2015 — Diabetes Health Indicators  
[UCI ML Repository · Dataset #891](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)

---

## Team

- Hasher Malik — 7632048
- Jan Erdorf — 8748557
- Ilias El Ouali — 000001
- Sophia Schaal — 7229428

## 1 · Business Understanding

### Use Case

Diabetes affects roughly 1 in 10 adults in the United States and is frequently undiagnosed. The goal of this project is to build a binary classifier that predicts whether an individual is at risk of having diabetes or prediabetes, using only self-reported health and lifestyle survey responses. Such a model could support public-health screening programs: flagging high-risk individuals for follow-up clinical testing without requiring laboratory results.

### Why Error Costs Are Asymmetric

In a medical screening context the costs of errors are asymmetric. A **false negative** — telling a person with diabetes that they are low-risk — deprives them of timely diagnosis and treatment, a high clinical cost. A **false positive** — flagging a healthy person as at-risk — leads to an unnecessary but low-cost follow-up test. This asymmetry shapes how we define success for the classifier and informs the choice of evaluation metric and decision threshold. These decisions are formalised in NB01; see the respective notebooks for results.

### Metrics and Imbalance

The dataset is heavily imbalanced, with the minority positive class (prediabetes/diabetes) representing roughly 14 % of all records. Standard accuracy is misleading under such conditions. Appropriate metrics for the imbalanced setting — including threshold-independent ranking metrics and recall-oriented criteria — are selected and justified in NB01; see the respective notebooks for results.

## 2 · The Data

### Source

The **Behavioral Risk Factor Surveillance System (BRFSS)** is an annual, state-based, random-digit-dialed telephone health survey conducted by the U.S. Centers for Disease Control and Prevention (CDC). It is one of the largest continuously running health surveys in the world, tracking behavioral risk factors and chronic conditions across U.S. adults. The 2015 wave of this survey is the basis for this project. The **“CDC Diabetes Health Indicators”** subset (UCI ML Repository, dataset #891) is derived from BRFSS 2015 and can be loaded programmatically via the `ucimlrepo` package.

### Structure

| Property | Value |
|---|---|
| Samples | ~253,680 |
| Features | 21 |
| Target | `Diabetes_binary` (0 = No Diabetes, 1 = Prediabetes / Diabetes) |
| Class split | ~86 % negative / ~14 % positive |
| Missing values | None (verified in NB02) |

Feature types: 14 binary, 4 ordinal (GenHlth, Age, Education, Income), 2 count (MentHlth, PhysHlth), 1 continuous (BMI).

### Survey Nature and Coarse Coding

All variables are self-reported responses to a telephone survey. Several continuous quantities (age, income, education, general health) are binned into ordinal scales. BMI is reported by respondents and not clinically measured. These properties limit the precision of individual features but are representative of the information available in real-world public-health screening scenarios.

### Key Challenges

**Class Imbalance.** The ~14 % positive rate biases naive classifiers and inflates accuracy-based metrics. Imbalance-handling strategies are evaluated and selected during the modelling phase (NB06).

**Label Noise / Positive-Unlabeled Structure.** The target records whether a respondent was *told by a doctor* that they have diabetes — it reflects diagnosis status, not disease status. Undiagnosed individuals appear as negatives despite potentially having the condition. This creates asymmetric label noise on the negative class, meaning measured performance metrics are a lower bound on true discriminative ability. This challenge is analysed in NB02 and discussed further in NB08.

**Exact Duplicate Rows.** The raw dataset contains a non-trivial number of exact duplicate survey responses. Naive random splitting risks placing the same row in both train and test, inflating test performance. This is handled at the split stage in NB03 (deduplication before stratified split) and analysed in NB02.

## 3 · Methodology — CRISP-DM

The project follows the **Cross-Industry Standard Process for Data Mining (CRISP-DM)**, an iterative, phase-structured framework for applied ML projects (Chapman et al., 2000). CRISP-DM is explicitly **iterative**: insights gained in later phases regularly feed back into earlier ones — for example, modelling difficulties may prompt a return to data preparation, or evaluation findings may reopen business-understanding questions. The cycle repeats until the solution meets the defined success criteria.

The six phases and their roles in this project are:

1. **Business Understanding** — define the problem, success criteria, and evaluation metrics from a domain perspective; establish the cost structure of prediction errors and the constraints on model behaviour.
2. **Data Understanding** — explore the dataset, assess quality and completeness, and identify potential modelling challenges such as class imbalance, label noise, and duplicate records.
3. **Data Preparation** — clean, transform, and engineer features; perform the train/test split in a leakage-free manner; prepare data for modelling pipelines.
4. **Modelling** — select candidate algorithms, build training pipelines (including imbalance-handling strategies), tune hyperparameters, and compare model performance under cross-validation.
5. **Evaluation** — assess the final model against business criteria on held-out test data; analyse fairness across demographic subgroups; produce calibration and explainability results.
6. **Deployment** — describe how the model would be operationalised in a public-health screening context (addressed conceptually within this seminar scope).

In this project the data preparation and modelling phases are each split across multiple notebooks to keep individual files focused. The diagram below (own rendering) illustrates the CRISP-DM cycle.

### Notebook–Phase Mapping

| Notebook | Title | CRISP-DM Phase |
|---|---|---|
| `01_business_understanding.ipynb` | Business Understanding | Business Understanding |
| `02_data_understanding.ipynb` | Data Understanding / EDA | Data Understanding |
| `03_data_preparation.ipynb` | Data Preparation & Split | Data Preparation |
| `04_feature_diagnostics.ipynb` | Feature Diagnostics | Data Preparation |
| `05_feature_engineering.ipynb` | Feature Engineering | Data Preparation |
| `06_modeling.ipynb` | Modelling — Baseline & Comparison | Modelling |
| `07_model_optimization.ipynb` | Model Optimisation & Selection | Modelling |
| `08_evaluation.ipynb` | Evaluation, Fairness & Explainability | Evaluation |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(-2.1, 2.1)
ax.set_ylim(-2.1, 2.1)
ax.set_aspect("equal")
ax.axis("off")
fig.patch.set_facecolor("white")

phases = [
    "Business\nUnderstanding",
    "Data\nUnderstanding",
    "Data\nPreparation",
    "Modelling",
    "Evaluation",
    "Deployment",
]

n = 6
radius = 1.35
angles = [np.pi / 2 - i * (2 * np.pi / n) for i in range(n)]
pos = np.array([(radius * np.cos(a), radius * np.sin(a)) for a in angles])

bw, bh = 0.58, 0.27
phase_color = "#2E75B6"


def edge_pts(p_from, p_to, offset=0.34):
    d = np.array(p_to) - np.array(p_from)
    dn = d / np.linalg.norm(d)
    return np.array(p_from) + dn * offset, np.array(p_to) - dn * offset


# Main clockwise flow
for i in range(n):
    s, e = edge_pts(pos[i], pos[(i + 1) % n])
    ax.annotate("", xy=e, xytext=s,
                arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#555555",
                                connectionstyle="arc3,rad=0.15"),
                zorder=2)

# Feedback arrows: DU->BU, Mod->DP, Eval->BU
feedback = [
    (1, 0, -0.30),   # Data Understanding -> Business Understanding
    (3, 2, -0.30),   # Modelling -> Data Preparation
    (4, 0, -0.50),   # Evaluation -> Business Understanding
]
for src_i, dst_i, rad in feedback:
    s, e = edge_pts(pos[src_i], pos[dst_i])
    ax.annotate("", xy=e, xytext=s,
                arrowprops=dict(arrowstyle="-|>", lw=1.3, color="#C00000",
                                linestyle="dashed",
                                connectionstyle=f"arc3,rad={rad}"),
                zorder=2)

# Central Data node
ax.add_patch(mpatches.Circle((0, 0), 0.26,
                              facecolor="#1F3864", edgecolor="white",
                              linewidth=2.5, zorder=4))
ax.text(0, 0, "Data", ha="center", va="center",
        fontsize=11, color="white", fontweight="bold", zorder=5)

# Phase boxes
for phase, (x, y) in zip(phases, pos):
    ax.add_patch(mpatches.FancyBboxPatch(
        (x - bw / 2, y - bh / 2), bw, bh,
        boxstyle="round,pad=0.04",
        facecolor=phase_color, edgecolor="white",
        linewidth=2, zorder=3))
    ax.text(x, y, phase, ha="center", va="center",
            fontsize=8.5, color="white", fontweight="bold",
            linespacing=1.3, zorder=4)

# Legend
lx, ly = -2.0, -1.85
ax.annotate("", xy=(lx + 0.45, ly), xytext=(lx, ly),
            arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#555555"))
ax.text(lx + 0.55, ly, "main flow", va="center", fontsize=8, color="#555555")
ax.annotate("", xy=(lx + 0.45, ly - 0.22), xytext=(lx, ly - 0.22),
            arrowprops=dict(arrowstyle="-|>", lw=1.3, color="#C00000",
                            linestyle="dashed"))
ax.text(lx + 0.55, ly - 0.22, "feedback", va="center", fontsize=8, color="#C00000")

ax.set_title("CRISP-DM Process Model  \u00b7  own rendering",
             fontsize=11, pad=10, color="#1F3864")
plt.tight_layout()
plt.show()

## 4 · Reproducibility & How to Run

### Environment

Install all dependencies from the repository root:

```bash
pip install -r requirements.txt
```

Python 3.10 or later is required. Key packages include `scikit-learn`, `lightgbm`, `catboost`, `optuna`, `imbalanced-learn`, `shap`, and `ucimlrepo`.

### Random Seed

Every notebook declares `SEED = 42` at the top and passes it to all stochastic operations (train/test split, cross-validation, model initialisation, Optuna sampler). No notebook introduces additional seeds.

### Data Access

Raw data is **not** stored in the repository. NB03 fetches it automatically:

```python
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=891)
```

NB03 then writes the stratified 80/20 split to `data/processed/` as Parquet files. All subsequent notebooks load these files. An internet connection is required for the first run.

### Run Order

Notebooks must be executed in numerical order:

```
01 → 02 → 03 → 04 → 05 → 06 → 07 → 08
```

NB03 produces the split artefacts that NB04–NB08 depend on. Running NB04 or later without first running NB03 will raise a `FileNotFoundError`.

### Repository Layout

```
diabetes-prediction-ml/
├── notebooks/          # one notebook per CRISP-DM step
├── src/
│   └── utils.py        # shared helpers (build_enriched_features, cap_bmi, …)
├── data/               # git-ignored; generated by NB03
│   ├── raw/
│   └── processed/
├── models/             # git-ignored; written by NB07
├── outputs/            # git-ignored; plots/CSVs per notebook
├── catboost_info/      # git-ignored; CatBoost training logs
├── requirements.txt
├── README.md
└── .gitignore
```

The directories `data/`, `models/`, `outputs/`, and `catboost_info/` are listed in `.gitignore` and are not tracked by Git.

In [ ]:
import sys
import platform

SEED = 42

print(f"Python  : {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"SEED    : {SEED}")
print()

_packages = [
    ("numpy",           "numpy"),
    ("pandas",          "pandas"),
    ("scikit-learn",    "sklearn"),
    ("catboost",        "catboost"),
    ("imbalanced-learn","imblearn"),
    ("ucimlrepo",       "ucimlrepo"),
]

for display, import_name in _packages:
    try:
        mod = __import__(import_name)
        version = getattr(mod, "__version__", "installed (version unknown)")
        print(f"{display:<20}: {version}")
    except ImportError:
        print(f"{display:<20}: NOT INSTALLED")

## References

Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc.

U.S. Centers for Disease Control and Prevention (CDC). *Behavioral Risk Factor Surveillance System (BRFSS), 2015 Survey Data*. Atlanta, GA: CDC.

UCI Machine Learning Repository. *CDC Diabetes Health Indicators* (Dataset #891). Original curation by A. Teboul (Kaggle). https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators